In [ ]:
# =============================================================================
# YouTube NLP Analizi – Türkiye Dijitalleşme Algısı (Gelişmiş)
# Google Colab'da çalışacak şekilde tasarlanmıştır.
# =============================================================================

# ── 0. KURULUM (Colab'da ilk hücre olarak çalıştırın) ────────────────────────
# !pip install -q google-api-python-client transformers torch
# !pip install -q bertopic umap-learn hdbscan
# !pip install -q wordcloud gensim pyLDAvis zeyrek snowballstemmer
# !pip install -q scikit-learn pandas matplotlib seaborn

import os, re, time, warnings, json
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")           # Colab'da ekran yoksa Agg backend kullan
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from collections import Counter
from datetime import datetime

# NLP
import nltk
nltk.download("stopwords", quiet=True)
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from gensim import corpora
from gensim.models import LdaModel
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# Stemming (TurkishStemmer – zeyrek kuruluysa lemmatizer da kullanılabilir)
try:
    from snowballstemmer import stemmer as SnowballStemmer
    TR_STEMMER = SnowballStemmer("turkish")
    USE_STEMMER = True
except ImportError:
    USE_STEMMER = False
    print("[UYARI] snowballstemmer bulunamadı; stemming atlanacak.")

# Transformers (BERTurk duygu)
try:
    from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
    TRANSFORMERS_OK = True
except ImportError:
    TRANSFORMERS_OK = False
    print("[UYARI] transformers bulunamadı; duygu analizi kural tabanlı çalışacak.")

# BERTopic
try:
    from bertopic import BERTopic
    from umap import UMAP
    from hdbscan import HDBSCAN
    BERTOPIC_OK = True
except ImportError:
    BERTOPIC_OK = False
    print("[UYARI] BERTopic bulunamadı; bu analiz atlanacak.")

# YouTube API
from googleapiclient.discovery import build

# ── 1. KONFİGÜRASYON ─────────────────────────────────────────────────────────
YOUTUBE_API_KEY       = "api"
MAX_VIDEOS_PER_KW     = 5
MAX_COMMENTS_PER_VID  = 200
RAW_CSV               = "youtube_ham.csv"
CLEAN_CSV             = "youtube_temiz.csv"
SENTIMENT_CSV         = "youtube_duygu.csv"
LDA_NUM_TOPICS        = 6
LDA_PASSES            = 15
TOP_N_WORDS           = 30
NGRAM_TOP_N           = 20
TFIDF_TOP_N           = 25
OUTPUT_DIR            = "gorseller"
os.makedirs(OUTPUT_DIR, exist_ok=True)

KEYWORDS = [
    "dijitalleşme Türkiye",
    "dijital uçurum",
    "e-devlet",
    "dijital eğitim",
    "yapay zeka Türkiye",
    "teknoloji Türkiye",
    "Türkiye dijital dönüşüm",
]

TURKISH_STOPWORDS = set([
    "bir","bu","ve","ile","da","de","için","ne","o","çok","daha","ama","ya",
    "ki","mi","mu","mı","mü","var","yok","en","gibi","kadar","benim","sen",
    "siz","biz","onlar","ben","onun","bunu","bunun","şu","şey","ise","den",
    "dan","te","ta","ye","ya","nun","nın","nin","nun","olan","olarak","edildi",
    "oldu","olup","veya","hem","bile","ancak","fakat","lakin","çünkü","eğer",
    "her","hiç","artık","zaten","sadece","yani","ayrıca","öyle","böyle",
    "nasıl","neden","nerede","hangi","bazı","tüm","hep","hepsi","diğer",
    "https","http","www","com","tr","co","bu","şu","o","ben","sen","biz","siz",
    "onlar","ben","beni","bana","bende","benden","benim","sen","seni","sana",
    "sende","senden","senin","o","onu","ona","onda","ondan","onun","biz","bizi",
    "bize","bizde","bizden","bizim","siz","sizi","size","sizde","sizden","sizin",
    "onlar","onları","onlara","onlarda","onlardan","onların","ki","de","da",
    "ile","için","gibi","kadar","diye","daha","en","çok","az","tam","hiç","bile",
    "ya","veya","yahut","ama","fakat","lakin","ancak","hem","ne","mi","mu","mı",
    "mü","var","yok","değil","olan","olarak","oldu","olup","edildi","yapıldı",
    "etti","etmek","olmak","yapmak","bu","şu","o","böyle","öyle","şöyle","nasıl",
    "neden","niçin","ne","nere","nerede","nereden","nereye","kim","kimi","kime",
    "kimde","kimden","kimin","hangi","hangisi","kaç","kaçıncı","ne","her","hiç",
    "tüm","bütün","bazı","birkaç","birçok","pek","epey","oldukça","gayet",
])

# ── 2. VERİ TOPLAMA ───────────────────────────────────────────────────────────
def get_video_ids(youtube, keyword, max_results=5):
    try:
        resp = youtube.search().list(
            q=keyword, part="id,snippet", type="video",
            relevanceLanguage="tr", regionCode="TR",
            maxResults=max_results, order="relevance",
        ).execute()
        return [(item["id"]["videoId"], item["snippet"]["title"])
                for item in resp.get("items", [])]
    except Exception as e:
        print(f"  [HATA] Arama: {e}")
        return []


def get_comments(youtube, video_id, video_title, keyword, max_results=200):
    rows, page_token = [], None
    while len(rows) < max_results:
        try:
            req = youtube.commentThreads().list(
                part="snippet", videoId=video_id,
                maxResults=min(100, max_results - len(rows)),
                textFormat="plainText", order="relevance",
                pageToken=page_token,
            )
            resp = req.execute()
            for item in resp.get("items", []):
                snip = item["snippet"]["topLevelComment"]["snippet"]
                rows.append({
                    "video_id"    : video_id,
                    "video_baslik": video_title,
                    "keyword"     : keyword,
                    "yorum"       : snip["textDisplay"],
                    "begeni"      : snip.get("likeCount", 0),
                    "yanit_sayisi": item["snippet"].get("totalReplyCount", 0),
                    "tarih"       : snip["publishedAt"],
                })
            page_token = resp.get("nextPageToken")
            if not page_token:
                break
        except Exception as e:
            print(f"  [UYARI] Yorum çekme ({video_id}): {e}")
            break
    return rows


def collect_data():
    if os.path.exists(RAW_CSV):
        print(f"[*] Mevcut CSV yükleniyor: '{RAW_CSV}'")
        return pd.read_csv(RAW_CSV, parse_dates=["tarih"])

    youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)
    all_rows, seen = [], set()

    for kw in KEYWORDS:
        print(f"\n[*] '{kw}'")
        videos = get_video_ids(youtube, kw, MAX_VIDEOS_PER_KW)
        for vid, title in videos:
            if vid in seen:
                continue
            seen.add(vid)
            rows = get_comments(youtube, vid, title, kw, MAX_COMMENTS_PER_VID)
            all_rows.extend(rows)
            print(f"  → {vid}: {len(rows)} yorum")
            time.sleep(0.4)

    df = pd.DataFrame(all_rows)
    df["tarih"] = pd.to_datetime(df["tarih"], errors="coerce", utc=True)
    df.drop_duplicates(subset="yorum", inplace=True)
    df.to_csv(RAW_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✓ Ham veri: {len(df)} yorum → '{RAW_CSV}'")
    return df


# ── 3. METİN ÖN İŞLEME ───────────────────────────────────────────────────────
EMOJI_RE    = re.compile("["
    u"\U0001F600-\U0001F64F"
    u"\U0001F300-\U0001F5FF"
    u"\U0001F680-\U0001F6FF"
    u"\U0001F1E0-\U0001F1FF"
    u"\U00002500-\U00002BEF"
    u"\U00002702-\U000027B0"
    u"\U000024C2-\U0001F251"
    "]+", flags=re.UNICODE)

def clean_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "", text)          # URL
    text = EMOJI_RE.sub("", text)                        # Emoji
    text = re.sub(r"[^\w\sğüşıöçĞÜŞİÖÇ]", " ", text)   # Noktalama (Türkçe koruma)
    text = re.sub(r"\d+", "", text)                      # Sayılar
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split()
              if t not in TURKISH_STOPWORDS and len(t) > 2]
    if USE_STEMMER:
        tokens = [TR_STEMMER.stemWord(t) for t in tokens]
    return " ".join(tokens)


def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.dropna(subset=["yorum"], inplace=True)
    df["temiz_yorum"] = df["yorum"].apply(clean_text)
    df = df[df["temiz_yorum"].str.strip() != ""]
    df.drop_duplicates(subset="temiz_yorum", inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"✓ Temizleme sonrası: {len(df)} yorum")
    return df


# ── 4. DUYGU ANALİZİ (BERTurk) ───────────────────────────────────────────────
BERTURK_MODEL = "savasy/bert-base-turkish-sentiment-cased"

POZITIF = {"güzel","harika","mükemmel","başarılı","iyi","süper","teşekkür",
            "olumlu","faydalı","yararlı","gelişme","ilerleme","umut","verimli",
            "etkili","memnun","mutlu","destekliyorum","avantaj","fırsat","başarı",
            "kaliteli","modern","güvenli","doğru","sevindim","etkileyici","kolay"}
NEGATIF = {"kötü","berbat","rezalet","sorun","problem","hata","yavaş","zor",
            "imkansız","korkunç","üzücü","başarısız","yetersiz","eksik","geri",
            "eşitsizlik","uçurum","engel","dezavantaj","kaygı","endişe","tehlike",
            "yanlış","haksız","mahrum","pahalı","zorluk","sıkıntı","bozuk",
            "çöktü","işe yaramıyor","negatif","olumsuz","manipülasyon"}

def rule_based_sentiment(text: str) -> str:
    tokens = set(text.split())
    p = len(tokens & POZITIF)
    n = len(tokens & NEGATIF)
    return "Pozitif" if p > n else ("Negatif" if n > p else "Nötr")


def run_sentiment(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if TRANSFORMERS_OK:
        print("[*] BERTurk duygu modeli yükleniyor (ilk kez yavaş olabilir)...")
        try:
            tokenizer = AutoTokenizer.from_pretrained(BERTURK_MODEL)
            model     = AutoModelForSequenceClassification.from_pretrained(BERTURK_MODEL)
            sentiment_pipe = pipeline(
                "text-classification", model=model, tokenizer=tokenizer,
                truncation=True, max_length=512,
            )
            label_map = {"positive": "Pozitif", "negative": "Negatif", "neutral": "Nötr",
                         "LABEL_0": "Negatif", "LABEL_1": "Nötr", "LABEL_2": "Pozitif"}

            results = []
            batch_size = 32
            texts = df["temiz_yorum"].tolist()
            for i in range(0, len(texts), batch_size):
                batch = texts[i:i+batch_size]
                preds = sentiment_pipe(batch)
                for pred in preds:
                    lbl = label_map.get(pred["label"], "Nötr")
                    results.append({"duygu": lbl, "duygu_skor": round(pred["score"], 4)})
                if i % 200 == 0:
                    print(f"  {i}/{len(texts)}", end="\r")

            sent_df = pd.DataFrame(results, index=df.index)
            df = pd.concat([df, sent_df], axis=1)
            print(f"\n✓ BERTurk duygu analizi tamamlandı.")
        except Exception as e:
            print(f"[UYARI] BERTurk hatası ({e}); kural tabanlı kullanılıyor.")
            df["duygu"]      = df["temiz_yorum"].apply(rule_based_sentiment)
            df["duygu_skor"] = 0.0
    else:
        df["duygu"]      = df["temiz_yorum"].apply(rule_based_sentiment)
        df["duygu_skor"] = 0.0

    print(df["duygu"].value_counts().to_string())
    return df


# ── 5. KELİME FREKANS ANALİZİ ────────────────────────────────────────────────
def word_frequency(df: pd.DataFrame, top_n=TOP_N_WORDS) -> list:
    words = " ".join(df["temiz_yorum"]).split()
    return Counter(words).most_common(top_n)


# ── 6. N-GRAM ANALİZİ ────────────────────────────────────────────────────────
def ngram_analysis(df: pd.DataFrame, n=2, top_n=NGRAM_TOP_N) -> list:
    vec = CountVectorizer(ngram_range=(n, n), max_features=5000)
    X   = vec.fit_transform(df["temiz_yorum"])
    totals = X.sum(axis=0).A1
    terms  = vec.get_feature_names_out()
    return sorted(zip(terms, totals), key=lambda x: -x[1])[:top_n]


# ── 7. TF-IDF ANALİZİ ────────────────────────────────────────────────────────
def tfidf_analysis(df: pd.DataFrame, top_n=TFIDF_TOP_N) -> pd.DataFrame:
    vec   = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    X     = vec.fit_transform(df["temiz_yorum"])
    means = X.mean(axis=0).A1
    terms = vec.get_feature_names_out()
    result = pd.DataFrame({"terim": terms, "tfidf_ortalama": means})
    return result.sort_values("tfidf_ortalama", ascending=False).head(top_n)


# ── 8. LDA KONU MODELLEMESİ ──────────────────────────────────────────────────
def run_lda(df: pd.DataFrame):
    tokenized  = [t.split() for t in df["temiz_yorum"]]
    dictionary = corpora.Dictionary(tokenized)
    dictionary.filter_extremes(no_below=3, no_above=0.85)
    corpus     = [dictionary.doc2bow(doc) for doc in tokenized]
    lda_model  = LdaModel(
        corpus=corpus, id2word=dictionary,
        num_topics=LDA_NUM_TOPICS, passes=LDA_PASSES,
        random_state=42, alpha="auto",
    )
    return lda_model, corpus, dictionary


def lda_dominant_topic(lda_model, corpus, df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    topics, scores = [], []
    for bow in corpus:
        dist = lda_model.get_document_topics(bow)
        if dist:
            top = max(dist, key=lambda x: x[1])
            topics.append(top[0] + 1)
            scores.append(round(top[1], 4))
        else:
            topics.append(0)
            scores.append(0.0)
    df["lda_konu"]  = topics
    df["lda_skor"]  = scores
    return df


# ── 9. BERTopic KONU MODELLEMESİ ─────────────────────────────────────────────
def run_bertopic(df: pd.DataFrame):
    if not BERTOPIC_OK:
        print("[UYARI] BERTopic yüklü değil; atlanıyor.")
        return None, None
    print("[*] BERTopic çalıştırılıyor...")
    umap_model  = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                       metric="cosine", random_state=42)
    hdbscan_mod = HDBSCAN(min_cluster_size=10, metric="euclidean",
                           cluster_selection_method="eom", prediction_data=True)
    topic_model = BERTopic(
        umap_model=umap_model, hdbscan_model=hdbscan_mod,
        language="multilingual", calculate_probabilities=True,
        verbose=False,
    )
    docs   = df["temiz_yorum"].tolist()
    topics, probs = topic_model.fit_transform(docs)
    return topic_model, topics


# ── 10. TEMSİLİ YORUMLAR (TEMA ÇIKARIMI) ─────────────────────────────────────
def representative_comments(df: pd.DataFrame, n=3) -> pd.DataFrame:
    rows = []
    for duygu in ["Pozitif", "Negatif", "Nötr"]:
        subset = df[df["duygu"] == duygu].copy()
        subset = subset.sort_values("begeni", ascending=False).head(n)
        for _, r in subset.iterrows():
            rows.append({
                "duygu"  : duygu,
                "yorum"  : r["yorum"][:200],
                "begeni" : r["begeni"],
                "keyword": r.get("keyword", ""),
            })
    return pd.DataFrame(rows)


# ── 11. GÖRSELLEŞTİRME ───────────────────────────────────────────────────────
PALETTE = {"Pozitif": "#16A34A", "Nötr": "#D97706", "Negatif": "#DC2626"}
C5      = ["#2563EB", "#DC2626", "#16A34A", "#D97706", "#7C3AED"]

def savefig(name: str):
    path = os.path.join(OUTPUT_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✓ {path}")


# 11-A  Duygu Dağılımı
def plot_sentiment(df: pd.DataFrame):
    counts = df["duygu"].value_counts()
    colors = [PALETTE.get(l, "#999") for l in counts.index]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Duygu Dağılımı", fontweight="bold", fontsize=14)

    ax1.pie(counts, labels=counts.index, autopct="%1.1f%%",
            colors=colors, startangle=140, textprops={"fontsize": 11})
    ax1.set_title("Pasta Grafik")

    bars = ax2.bar(counts.index, counts.values, color=colors, width=0.5)
    for b in bars:
        ax2.text(b.get_x() + b.get_width() / 2, b.get_height() + 2,
                 str(int(b.get_height())), ha="center", fontsize=10)
    ax2.set_ylabel("Yorum Sayısı")
    ax2.set_title("Bar Grafik")

    plt.tight_layout()
    savefig("01_duygu_dagilimi.png")


# 11-B  Kelime Frekansı
def plot_word_freq(freq: list):
    words, counts = zip(*freq) if freq else ([], [])
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(list(words)[::-1], list(counts)[::-1], color=C5[0])
    ax.set_title(f"En Sık Kullanılan {TOP_N_WORDS} Kelime", fontweight="bold")
    ax.set_xlabel("Frekans")
    plt.tight_layout()
    savefig("02_kelime_frekansi.png")


# 11-C  Kelime Bulutu
def plot_wordcloud(df: pd.DataFrame):
    text = " ".join(df["temiz_yorum"])
    wc   = WordCloud(width=1200, height=600, background_color="white",
                     colormap="Blues", max_words=200).generate(text)
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title("Kelime Bulutu", fontweight="bold", fontsize=14)
    plt.tight_layout()
    savefig("03_kelime_bulutu.png")


# 11-D  N-gram Grafikleri
def plot_ngrams(bigrams: list, trigrams: list):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    for ax, grams, title in zip(axes,
                                 [bigrams, trigrams],
                                 ["En Sık Bigramlar", "En Sık Trigramlar"]):
        if not grams:
            continue
        labels, vals = zip(*grams)
        ax.barh(list(labels)[::-1], list(vals)[::-1], color=C5[1])
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Frekans")
        ax.tick_params(axis="y", labelsize=8)
    plt.suptitle("N-gram Analizi", fontweight="bold", fontsize=14)
    plt.tight_layout()
    savefig("04_ngram_analizi.png")


# 11-E  TF-IDF
def plot_tfidf(tfidf_df: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(tfidf_df["terim"][::-1], tfidf_df["tfidf_ortalama"][::-1], color=C5[2])
    ax.set_title("TF-IDF – En Yüksek Ortalama Skorlu Terimler", fontweight="bold")
    ax.set_xlabel("Ortalama TF-IDF Skoru")
    ax.tick_params(axis="y", labelsize=8)
    plt.tight_layout()
    savefig("05_tfidf_analizi.png")


# 11-F  LDA Konu Grafikleri
def plot_lda_topics(lda_model):
    n_topics = lda_model.num_topics
    ncols    = 2
    nrows    = (n_topics + 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 4))
    axes = axes.flatten()

    for t in range(n_topics):
        pairs   = lda_model.show_topic(t, topn=10)
        words   = [p[0] for p in pairs]
        weights = [p[1] for p in pairs]
        axes[t].barh(words[::-1], weights[::-1], color=C5[t % len(C5)])
        axes[t].set_title(f"Konu {t+1}", fontweight="bold")
        axes[t].set_xlabel("Ağırlık")
        axes[t].tick_params(axis="y", labelsize=8)

    for i in range(n_topics, len(axes)):
        axes[i].axis("off")

    plt.suptitle("LDA Konu Modelleme – Konu Başına Top Kelimeler",
                 fontweight="bold", fontsize=14)
    plt.tight_layout()
    savefig("06_lda_konular.png")


def plot_lda_dağılım(df: pd.DataFrame):
    counts = df["lda_konu"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar([f"Konu {k}" for k in counts.index], counts.values, color=C5)
    ax.set_title("LDA – Konu Başına Yorum Dağılımı", fontweight="bold")
    ax.set_ylabel("Yorum Sayısı")
    plt.tight_layout()
    savefig("07_lda_yorum_dagilimi.png")


# 11-G  Zaman Serisi
def plot_time_series(df: pd.DataFrame):
    if "tarih" not in df.columns or df["tarih"].isna().all():
        print("  [UYARI] Tarih verisi yok; zaman serisi atlanıyor.")
        return
    df2 = df.copy()
    df2["ay"] = df2["tarih"].dt.to_period("M").astype(str)

    # Aylık toplam
    monthly = df2.groupby("ay").size().reset_index(name="sayi")

    # Aylık duygu
    monthly_sent = (df2.groupby(["ay", "duygu"])
                       .size().unstack(fill_value=0).reset_index())

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    axes[0].bar(monthly["ay"], monthly["sayi"], color=C5[0])
    axes[0].set_title("Aylık Yorum Sayısı", fontweight="bold")
    axes[0].set_ylabel("Yorum Sayısı")
    axes[0].tick_params(axis="x", rotation=45, labelsize=7)

    for col in ["Pozitif", "Negatif", "Nötr"]:
        if col in monthly_sent.columns:
            axes[1].plot(monthly_sent["ay"], monthly_sent[col],
                         label=col, color=PALETTE.get(col), marker="o", markersize=4)
    axes[1].set_title("Aylık Duygu Dağılımı", fontweight="bold")
    axes[1].set_ylabel("Yorum Sayısı")
    axes[1].tick_params(axis="x", rotation=45, labelsize=7)
    axes[1].legend()

    plt.suptitle("Zaman Serisi Analizi", fontweight="bold", fontsize=14)
    plt.tight_layout()
    savefig("08_zaman_serisi.png")


# 11-H  En Çok Etkileşim Alan Yorumlar
def plot_top_engagement(df: pd.DataFrame, top_n=15):
    top = df.nlargest(top_n, "begeni")[["yorum", "begeni", "duygu"]].copy()
    top["yorum_kisa"] = top["yorum"].str[:50] + "…"
    colors = [PALETTE.get(d, "#999") for d in top["duygu"]]

    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(top["yorum_kisa"][::-1], top["begeni"][::-1], color=colors[::-1])
    ax.set_title(f"En Çok Beğenilen {top_n} Yorum", fontweight="bold")
    ax.set_xlabel("Beğeni Sayısı")
    ax.tick_params(axis="y", labelsize=8)
    plt.tight_layout()
    savefig("09_top_engagement.png")


# 11-I  Duygu × Anahtar Kelime Isı Haritası
def plot_sentiment_heatmap(df: pd.DataFrame):
    pivot = (df.groupby(["keyword", "duygu"])
               .size().unstack(fill_value=0))
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(pivot, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title("Anahtar Kelime × Duygu Isı Haritası", fontweight="bold")
    ax.set_xlabel("Duygu")
    ax.set_ylabel("Anahtar Kelime")
    plt.tight_layout()
    savefig("10_duygu_isı_haritasi.png")


# 11-J  BERTopic
def plot_bertopic(topic_model, topics, df: pd.DataFrame):
    if topic_model is None:
        return

    # Bar chart (topic info)
    info = topic_model.get_topic_info()
    info = info[info["Topic"] != -1].head(10)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(info["Name"].str[:40][::-1], info["Count"][::-1], color=C5[3])
    ax.set_title("BERTopic – En Büyük Konular", fontweight="bold")
    ax.set_xlabel("Yorum Sayısı")
    ax.tick_params(axis="y", labelsize=8)
    plt.tight_layout()
    savefig("11_bertopic_konular.png")

    # İnteraktif HTML
    try:
        fig_html = topic_model.visualize_topics()
        fig_html.write_html(os.path.join(OUTPUT_DIR, "bertopic_interaktif.html"))
        fig_barchart = topic_model.visualize_barchart(top_n_topics=10)
        fig_barchart.write_html(os.path.join(OUTPUT_DIR, "bertopic_barchart.html"))
        print("  ✓ gorseller/bertopic_interaktif.html")
        print("  ✓ gorseller/bertopic_barchart.html")
    except Exception as e:
        print(f"  [UYARI] BERTopic HTML: {e}")


# ── 12. ÇIKTI KAYDETME ───────────────────────────────────────────────────────
def save_lda_output(lda_model, corpus, dictionary):
    rows = []
    for t in range(lda_model.num_topics):
        pairs = lda_model.show_topic(t, topn=15)
        rows.append({
            "konu_no"  : t + 1,
            "kelimeler": ", ".join([p[0] for p in pairs]),
            "agirliklar": ", ".join([str(round(p[1], 4)) for p in pairs]),
        })
    pd.DataFrame(rows).to_csv(os.path.join(OUTPUT_DIR, "lda_konular.csv"),
                               index=False, encoding="utf-8-sig")

    try:
        vis = gensimvis.prepare(lda_model, corpus, dictionary, mds="mmds")
        pyLDAvis.save_html(vis, os.path.join(OUTPUT_DIR, "lda_interaktif.html"))
        print("  ✓ gorseller/lda_interaktif.html")
    except Exception as e:
        print(f"  [UYARI] pyLDAvis: {e}")


def save_bertopic_output(topic_model):
    if topic_model is None:
        return
    info = topic_model.get_topic_info()
    info.to_csv(os.path.join(OUTPUT_DIR, "bertopic_konular.csv"),
                index=False, encoding="utf-8-sig")


def print_topic_summary(lda_model):
    print("\n──────────────────────────────────────────────")
    print("  LDA KONU ÖZETİ")
    print("──────────────────────────────────────────────")
    for i, topic in lda_model.print_topics(num_words=10):
        print(f"  Konu {i+1}: {topic}")
    print("──────────────────────────────────────────────")


# ── 13. ANA AKIŞ ─────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("  YouTube NLP – Türkiye Dijitalleşme Algısı (Gelişmiş)")
    print("=" * 60)

    # 13-1  Veri Toplama
    df_raw = collect_data()
    if df_raw.empty:
        print("[HATA] Yorum toplanamadı. API anahtarını kontrol edin.")
        return

    # 13-2  Temizleme
    print("\n[*] Metin ön işleme...")
    df = preprocess(df_raw)
    df.to_csv(CLEAN_CSV, index=False, encoding="utf-8-sig")
    print(f"✓ Temiz veri kaydedildi → '{CLEAN_CSV}'")

    # 13-3  Duygu Analizi
    print("\n[*] Duygu analizi...")
    df = run_sentiment(df)
    df.to_csv(SENTIMENT_CSV, index=False, encoding="utf-8-sig")
    print(f"✓ Duygu sonuçları kaydedildi → '{SENTIMENT_CSV}'")

    # 13-4  Analizler
    print("\n[*] Kelime frekansı...")
    freq = word_frequency(df)

    print("[*] N-gram analizi...")
    bigrams  = ngram_analysis(df, n=2)
    trigrams = ngram_analysis(df, n=3)

    print("[*] TF-IDF analizi...")
    tfidf_df = tfidf_analysis(df)
    tfidf_df.to_csv(os.path.join(OUTPUT_DIR, "tfidf_sonuclar.csv"),
                    index=False, encoding="utf-8-sig")

    print("[*] LDA konu modelleme...")
    lda_model, corpus, dictionary = run_lda(df)
    df = lda_dominant_topic(lda_model, corpus, df)
    print_topic_summary(lda_model)

    print("[*] BERTopic...")
    topic_model, topics = run_bertopic(df)

    # Temsilî yorumlar
    rep_df = representative_comments(df)
    rep_df.to_csv(os.path.join(OUTPUT_DIR, "temsili_yorumlar.csv"),
                  index=False, encoding="utf-8-sig")
    print("\n  Temsilî Yorumlar:")
    print(rep_df[["duygu", "yorum", "begeni"]].to_string(index=False))

    # 13-5  Görselleştirme
    print("\n[*] Görseller oluşturuluyor...")
    plot_sentiment(df)
    plot_word_freq(freq)
    plot_wordcloud(df)
    plot_ngrams(bigrams, trigrams)
    plot_tfidf(tfidf_df)
    plot_lda_topics(lda_model)
    plot_lda_dağılım(df)
    plot_time_series(df)
    plot_top_engagement(df)
    plot_sentiment_heatmap(df)
    plot_bertopic(topic_model, topics, df)

    # 13-6  Çıktı Kaydet
    print("\n[*] Çıktılar kaydediliyor...")
    save_lda_output(lda_model, corpus, dictionary)
    save_bertopic_output(topic_model)

    # Nihai CSV
    final_csv = os.path.join(OUTPUT_DIR, "youtube_nlp_tam.csv")
    df.to_csv(final_csv, index=False, encoding="utf-8-sig")
    print(f"✓ Nihai veri seti → '{final_csv}'")

    print("\n" + "=" * 60)
    print("  ✅ Tüm analizler tamamlandı.")
    print(f"  📁 Çıktılar: '{OUTPUT_DIR}/' klasöründe")
    print("=" * 60)


if __name__ == "__main__":
    main()

[UYARI] transformers bulunamadı; duygu analizi kural tabanlı çalışacak.
[UYARI] BERTopic bulunamadı; bu analiz atlanacak.
  YouTube NLP – Türkiye Dijitalleşme Algısı (Gelişmiş)
[*] Mevcut CSV yükleniyor: 'youtube_ham.csv'

[*] Metin ön işleme...
✓ Temizleme sonrası: 854 yorum
✓ Temiz veri kaydedildi → 'youtube_temiz.csv'

[*] Duygu analizi...
duygu
Nötr       737
Pozitif     99
Negatif     18
✓ Duygu sonuçları kaydedildi → 'youtube_duygu.csv'

[*] Kelime frekansı...
[*] N-gram analizi...
[*] TF-IDF analizi...
[*] LDA konu modelleme...

──────────────────────────────────────────────
  LDA KONU ÖZETİ
──────────────────────────────────────────────
  Konu 1: 0.094*"jax" + 0.039*"ap" + 0.033*"me" + 0.026*"ner" + 0.021*"cain" + 0.020*"ni" + 0.018*"yarar" + 0.013*"va" + 0.013*"gangle" + 0.012*"ada"
  Konu 2: 0.042*"ol" + 0.031*"güzel" + 0.016*"teşekkür" + 0.013*"video" + 0.012*"allah" + 0.012*"raz" + 0.011*"genç" + 0.011*"ülke" + 0.011*"eğit" + 0.011*"zeka"
  Konu 3: 0.023*"sağlık" + 0.016*"b